# Simple RAG in Colab: PDF → FAISS → Qwen2.5-3B
**Before you run:** Runtime → Change runtime type → **T4 GPU**.
Then run the cells top to bottom.

## 1. Install libraries

In [1]:
!pip -q install pymupdf sentence-transformers faiss-cpu

## 2. Get a PDF
Default: the original RAG paper (Lewis et al., 2020), downloaded from arXiv.
To use your own PDF instead, run the upload cell and skip the download cell.

In [2]:
import os
os.makedirs("data/documents", exist_ok=True)
!wget -q -O data/documents/rag_paper.pdf https://arxiv.org/pdf/2005.11401
!ls -lh data/documents

total 868K
-rw-r--r-- 1 root root 865K Jan 23  2023 rag_paper.pdf


## 3. Extract text page by page (keep source + page number)

In [3]:
from pathlib import Path
import fitz  # PyMuPDF

def extract_pdf_pages(pdf_path):
    pages = []
    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({"text": text, "source": Path(pdf_path).name, "page": page_number})
    return pages

all_pages = []
for pdf_path in Path("data/documents").glob("*.pdf"):
    all_pages.extend(extract_pdf_pages(pdf_path))

print("Total extracted pages:", len(all_pages))
print(all_pages[0]["text"][:500])   # quality check: does this look like real text?

Total extracted pages: 19
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research; ‡University College London; ⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## 4. Split into chunks (900 chars, 150 overlap)

In [4]:
def split_into_chunks(text, chunk_size=900, overlap=150):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = end - overlap
    return chunks

all_chunks = []
for page in all_pages:
    for chunk_id, chunk_text in enumerate(split_into_chunks(page["text"])):
        all_chunks.append({"text": chunk_text, "source": page["source"],
                           "page": page["page"], "chunk_id": chunk_id})

print("Total chunks:", len(all_chunks))

Total chunks: 99


## 5. Embed chunks and build the FAISS index

In [5]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

texts = [c["text"] for c in all_chunks]
embeddings = embedding_model.encode(
    texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
).astype("float32")
print("Embedding shape:", embeddings.shape)

index = faiss.IndexFlatIP(embeddings.shape[1])   # exact search, cosine (vectors are normalized)
index.add(embeddings)
print("Indexed vectors:", index.ntotal)   # chunk i in all_chunks == vector i in index

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding shape: (99, 384)
Indexed vectors: 99


## 6. Retrieval

In [6]:
def retrieve(question, top_k=4):
    q = embedding_model.encode([question], convert_to_numpy=True,
                               normalize_embeddings=True).astype("float32")
    scores, positions = index.search(q, top_k)
    results = []
    for score, pos in zip(scores[0], positions[0]):
        if pos == -1:
            continue
        item = dict(all_chunks[pos])
        item["score"] = float(score)
        results.append(item)
    return results

# Test retrieval alone before adding the LLM
for r in retrieve("What retriever does RAG use?"):
    print(f"{r['score']:.3f} | {r['source']} p.{r['page']}\n{r['text'][:200]}...\n")

0.656 | rag_paper.pdf p.5
RAG enjoys strong results
without expensive, specialized “salient span masking” pre-training [20]. It is worth noting that RAG’s
retriever is initialized using DPR’s retriever, which uses retrieval su...

0.569 | rag_paper.pdf p.7
igniﬁcantly more diverse than BART without needing
any diversity-promoting decoding.
Retrieval Ablations
A key feature of RAG is learning to retrieve relevant information for the task.
To assess the e...

0.381 | rag_paper.pdf p.9
rning latent retrieval, and retrieving evidence documents
rather than related training pairs. This said, RAG techniques may work well in these settings, and
could represent promising future work.
6
Di...

0.365 | rag_paper.pdf p.2
rator and retriever are jointly learned.
There has been extensive previous work proposing architectures to enrich systems with non-parametric
memory which are trained from scratch for speciﬁc tasks, e...



## 7. Load the language model (≈6 GB in fp16, fits a free T4)

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)
print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda:0


## 8. Ask questions (retrieve → prompt → generate)

In [8]:
def ask(question, top_k=4, max_new_tokens=300, show_sources=True):
    retrieved = retrieve(question, top_k=top_k)

    context = "\n\n".join(
        f"[{i}] Source: {r['source']}, page {r['page']}\n{r['text']}"
        for i, r in enumerate(retrieved, start=1)
    )
    messages = [
        {"role": "system", "content":
            "Answer using only the supplied context. If the context does not "
            "contain the answer, say there is not enough information. "
            "Refer to source numbers when possible."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print("QUESTION:", question)
    print("\nANSWER:", answer.strip())
    if show_sources:
        print("\nSOURCES:")
        for i, r in enumerate(retrieved, start=1):
            print(f"  [{i}] {r['source']} p.{r['page']} (score {r['score']:.3f})")
    return answer, retrieved

_ = ask("What are the two RAG formulations and how do they differ?")

QUESTION: What are the two RAG formulations and how do they differ?

ANSWER: The two RAG formulations mentioned are RAG-Token and RAG-Sequence. The difference between them lies in how they process input data; RAG-Token processes tokenized inputs whereas RAG-Sequence processes sequence inputs.

SOURCES:
  [1] rag_paper.pdf p.9 (score 0.516)
  [2] rag_paper.pdf p.5 (score 0.446)
  [3] rag_paper.pdf p.6 (score 0.418)
  [4] rag_paper.pdf p.17 (score 0.411)


In [9]:
_ = ask("Which retriever does RAG use, and what is its generator?")

QUESTION: Which retriever does RAG use, and what is its generator?

ANSWER: RAG uses a retriever that is initialized using DPR's retriever, which uses retrieval supervision on Natural Questions and TriviaQA. The generator of RAG is a model that generates answers.

SOURCES:
  [1] rag_paper.pdf p.5 (score 0.637)
  [2] rag_paper.pdf p.7 (score 0.565)
  [3] rag_paper.pdf p.2 (score 0.442)
  [4] rag_paper.pdf p.9 (score 0.415)


### Test an unanswerable question
A good RAG system should say it doesn't have enough information instead of inventing an answer.

In [10]:
_ = ask("What is the capital of France?")

QUESTION: What is the capital of France?

ANSWER: There is not enough information.

SOURCES:
  [1] rag_paper.pdf p.7 (score 0.207)
  [2] rag_paper.pdf p.5 (score 0.133)
  [3] rag_paper.pdf p.15 (score 0.121)
  [4] rag_paper.pdf p.14 (score 0.115)


In [13]:
_ = ask("give me a simple rag explanation")

QUESTION: give me a simple rag explanation

ANSWER: RAG, or Retrieval-Augmented Generation, combines a dense vector index of Wikipedia (non-parametric memory) with a pre-trained neural retriever and a seq2seq model (parametric memory). The retriever provides latent documents conditioned on the input, which the seq2seq model then uses to generate the output. This model can be fine-tuned on any seq2seq task, and it works by marginalizing the latent documents with a top-K approximation. It demonstrates how non-parametric and parametric memories work together, with the non-parametric component guiding the generation based on specific knowledge stored in the parametric memory.

SOURCES:
  [1] rag_paper.pdf p.2 (score 0.307)
  [2] rag_paper.pdf p.6 (score 0.299)
  [3] rag_paper.pdf p.4 (score 0.294)
  [4] rag_paper.pdf p.7 (score 0.294)
